## Data Cleaning

### 1. Load the Original Data

In [ ]:
# Load data
import pandas as pd

prices = pd.read_parquet("prices.parquet")
news = pd.read_parquet("news.parquet")
filings = pd.read_parquet("filings.parquet")

prices.head()

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600


### 2. Inspect Dataset Dimensions and Columns

In [11]:
# Check the number of rows, columns, and column names for each datase
print(prices.shape)
print(prices.columns)

print(news.shape)
print(news.columns)

print(filings.shape)
print(filings.columns)

(2656, 9)
Index(['id', 'symbol', 'date', 'open', 'high', 'low', 'close', 'adj_close',
       'volume'],
      dtype='str')
(3888, 4)
Index(['id', 'symbol', 'date', 'highlights'], dtype='str')
(73, 6)
Index(['id', 'symbol', 'date', 'document_type', 'mda_content', 'risk_content'], dtype='str')


### 3. Inspect Data Types

In [12]:
# Check the data type of each column
print(prices.dtypes)
print()
print(news.dtypes)
print()
print(filings.dtypes)

id             int32
symbol           str
date          object
open         float64
high         float64
low          float64
close        float64
adj_close    float64
volume         int64
dtype: object

id             int32
symbol           str
date          object
highlights       str
dtype: object

id                int32
symbol              str
date             object
document_type       str
mda_content         str
risk_content        str
dtype: object


### 4. Check Missing Values

In [ ]:
# Count missing values in each column
print("PRICES")
print(prices.isna().sum())

print("\nNEWS")
print(news.isna().sum())

print("\nFILINGS")
print(filings.isna().sum())

PRICES
id           0
symbol       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64

NEWS
id            0
symbol        0
date          0
highlights    0
dtype: int64

FILINGS
id               0
symbol           0
date             0
document_type    0
mda_content      0
risk_content     0
dtype: int64


### 5. Check Exact Duplicate Rows

In [13]:
# Count exact duplicate rows in each dataset
print("prices duplicates:", prices.duplicated().sum())
print("news duplicates:", news.duplicated().sum())
print("filings duplicates:", filings.duplicated().sum())

prices duplicates: 0
news duplicates: 0
filings duplicates: 0


### 6. Check Repeated IDs and Symbol-Date Combinations

In [9]:
print("prices repeated IDs:", prices["id"].duplicated().sum())
print("news repeated IDs:", news["id"].duplicated().sum())
print("filings repeated IDs:", filings["id"].duplicated().sum())

print(
    "prices repeated symbol-date:",
    prices.duplicated(subset=["symbol", "date"]).sum()
)

print(
    "news repeated symbol-date:",
    news.duplicated(subset=["symbol", "date"]).sum()
)

print(
    "filings repeated symbol-date:",
    filings.duplicated(subset=["symbol", "date"]).sum()
)

prices repeated IDs: 0
news repeated IDs: 0
filings repeated IDs: 0
prices repeated symbol-date: 0
news repeated symbol-date: 0
filings repeated symbol-date: 0


### 7. Check Suspicious Values


In [14]:
# Check for suspicious price and volume values

print("high < low:", (prices["high"] < prices["low"]).sum())

print("high < open:", (prices["high"] < prices["open"]).sum())
print("high < close:", (prices["high"] < prices["close"]).sum())

print("low > open:", (prices["low"] > prices["open"]).sum())
print("low > close:", (prices["low"] > prices["close"]).sum())

print("negative/zero volume:", (prices["volume"] <= 0).sum())

print("nonpositive open:", (prices["open"] <= 0).sum())
print("nonpositive high:", (prices["high"] <= 0).sum())
print("nonpositive low:", (prices["low"] <= 0).sum())
print("nonpositive close:", (prices["close"] <= 0).sum())

high < low: 0
high < open: 0
high < close: 0
low > open: 0
low > close: 0
negative/zero volume: 0
nonpositive open: 0
nonpositive high: 0
nonpositive low: 0
nonpositive close: 0


In [15]:
# Check whether all date values can be converted to valid dates

print("prices invalid dates:",
      pd.to_datetime(prices["date"], errors="coerce").isna().sum())

print("news invalid dates:",
      pd.to_datetime(news["date"], errors="coerce").isna().sum())

print("filings invalid dates:",
      pd.to_datetime(filings["date"], errors="coerce").isna().sum())

prices invalid dates: 0
news invalid dates: 0
filings invalid dates: 0


### 8. Define Cleaning Rules

**Cleaning Rules**
1. Convert the `date` column in all three datasets from object to datetime because the values are valid dates but are not stored as a datetime type.
2. Keep all rows because no missing values, exact duplicates, repeated IDs, or repeated symbol-date combinations were found.
3. Keep all price and volume values because no invalid or suspicious values were found.

### 9. Apply Cleaning Rules 

In [16]:
# Create copies of the original datasets
prices_clean = prices.copy()
news_clean = news.copy()
filings_clean = filings.copy()

# Convert date columns from object to datetime
prices_clean["date"] = pd.to_datetime(prices_clean["date"])
news_clean["date"] = pd.to_datetime(news_clean["date"])
filings_clean["date"] = pd.to_datetime(filings_clean["date"])

### 10. Verify Date Conversion

In [17]:
# Verify that date columns were converted to datetime

print("prices date type:", prices_clean["date"].dtype)
print("news date type:", news_clean["date"].dtype)
print("filings date type:", filings_clean["date"].dtype)

prices date type: datetime64[s]
news date type: datetime64[s]
filings date type: datetime64[s]


### 11. Reconcile Changes

In [18]:
# Compare row counts before and after cleaning

print("Prices:", len(prices), "->", len(prices_clean))
print("News:", len(news), "->", len(news_clean))
print("Filings:", len(filings), "->", len(filings_clean))

Prices: 2656 -> 2656
News: 3888 -> 3888
Filings: 73 -> 73


In [19]:
# Compare important totals before and after cleaning

print("Total volume before:", prices["volume"].sum())
print("Total volume after:", prices_clean["volume"].sum())

print("Total close before:", prices["close"].sum())
print("Total close after:", prices_clean["close"].sum())

Total volume before: 159456800900
Total volume after: 159456800900
Total close before: 894730.3596954346
Total close after: 894730.3596954346


### 12. Verify Individual Records

**Expected Results**

For five selected records, I expect the date value to remain the same while its data type changes from object to datetime. All other values in each record should remain unchanged.

In [20]:
# Select five records for verification
prices.iloc[[0, 1, 100, 1000, 2655]]

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
100,101,AAPL,2025-04-29,208.690002,212.240005,208.369995,211.210007,210.293457,36827600
1000,1001,GOOGL,2024-12-06,172.029999,175.080002,171.860001,174.710007,173.706955,21462400
2655,2656,TSLA,2026-03-31,361.510010,373.329987,361.000000,371.750000,371.750000,75534900


In [21]:
# Compare original and cleaned records

for i in [0, 1, 100, 1000, 2655]:
    print("Record:", i)
    print("Original date:", prices.loc[i, "date"])
    print("Cleaned date:", prices_clean.loc[i, "date"])
    print("Other values unchanged:",
          prices.loc[i].drop("date").equals(
              prices_clean.loc[i].drop("date")
          ))
    print()

Record: 0
Original date: 2024-12-02
Cleaned date: 2024-12-02 00:00:00
Other values unchanged: True

Record: 1
Original date: 2024-12-03
Cleaned date: 2024-12-03 00:00:00
Other values unchanged: True

Record: 100
Original date: 2025-04-29
Cleaned date: 2025-04-29 00:00:00
Other values unchanged: True

Record: 1000
Original date: 2024-12-06
Cleaned date: 2024-12-06 00:00:00
Other values unchanged: True

Record: 2655
Original date: 2026-03-31
Cleaned date: 2026-03-31 00:00:00
Other values unchanged: True



**Verification Result**

All five selected records matched the expected results. The date values were preserved and successfully converted to datetime, while all other values remained unchanged.

### 13. Test Reproducibility